In [33]:
import requests

def _prebuilt_placeholder(tool_data):
    def _run(*args, **kwargs):
        return {
            "tool_type": "prebuilt",
            "tool_id": tool_data["id"],
            "message": "Prebuilt tool execution placeholder"
        }
    return _run

def _custom_function_placeholder(tool_data):
    def _run(*args, **kwargs):
        pass
    return _run

In [34]:
def _custom_api_placeholder(tool_data):
    input_schema = tool_data.get("input_schema") or {}
    output_schema = tool_data.get("output_schema") or {}
    allowed_fields = set(input_schema.get("properties", {}).keys())
    
    print(input_schema)
    def _run(tool_input=None, **kwargs):
        payload = {}
        if isinstance(tool_input, dict):
            payload.update(tool_input)
        payload.update({k: v for k, v in kwargs.items() if k in allowed_fields})

        api_url = tool_data.get("api_url")
        method = (tool_data.get("api_request_type") or "GET").upper()
        custom_message = tool_data.get("custom_message")
        timeout = tool_data.get("timeout", 10)

        if not api_url:
            return {"status_code": None, "output": None, "custom_message": custom_message, "error": "Missing api_url"}

        try:
            if method == "GET":
                response = requests.get(api_url, params=payload, timeout=timeout)
            elif method == "POST":
                response = requests.post(api_url, json=payload, timeout=timeout)
            else:
                return {"status_code": None, "output": None, "custom_message": custom_message, "error": f"Unsupported method {method}"}

            try:
                raw_output = response.json()
            except ValueError:
                raw_output = response.text if response.text else None

            if isinstance(raw_output, dict) and "properties" in output_schema:
                allowed_outputs = output_schema["properties"].keys()
                shaped_output = {k: raw_output.get(k) for k in allowed_outputs}
            else:
                shaped_output = raw_output

            return {"status_code": response.status_code, "output": shaped_output, "custom_message": custom_message}

        except requests.RequestException as exc:
            return {"status_code": None, "output": None, "custom_message": custom_message, "error": str(exc)}

    return _run

In [35]:
from typing import List
from langchain_core.tools import Tool

from manager import ToolRegistryManager

def build_langchain_tools(tool_ids: List[str]) -> List[Tool]:
    """
    Given a list of tool IDs, return LangChain-compatible Tool objects.
    """
    manager = ToolRegistryManager()
    langchain_tools: List[Tool] = []

    for tool_id in tool_ids:
        tool_data = manager.get_tool(tool_id)
        if not tool_data:
            print("tool is missing")

        tool_type = tool_data.get("type")
        name = tool_data.get("name")
        description = tool_data.get("description")

        # --- Placeholder execution functions ---
        if tool_type == "prebuilt":
            func = _prebuilt_placeholder(tool_data)

        elif tool_type == "custom_function":
            func = _custom_function_placeholder(tool_data)

        elif tool_type == "custom_api":
            func = _custom_api_placeholder(tool_data)

        else:
            continue

        langchain_tools.append(
            Tool(
                name=name,
                description=description,
                func=func
            )
        )

    return langchain_tools

In [36]:
tool_ids = [
    "agent_8b8af64a-5063-4eb3-a00c-86c40e74ce43",
    "agent_84f2d4f0-97ad-457c-9f8b-6a70c8eb80af",
    "agent_f35d529c-5c1a-4bf5-9959-f501cbbdd9b3"
]

tools = build_langchain_tools(tool_ids)

{'type': 'object'}


In [16]:
response = tools[2].invoke({"user_id": "123"})
response

{'status_code': None,
 'output': None,
 'custom_message': 'take only the order id and provide it to the user',
 'error': 'HTTPConnectionPool(host=\'127.0.0.1\', port=8000): Max retries exceeded with url: /v1/custom-api (Caused by NewConnectionError("HTTPConnection(host=\'127.0.0.1\', port=8000): Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it"))'}

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class UserQuery(BaseModel):
    user_id: int = Field(..., description="ID of the user")

def user_info(user_id: int) -> dict:
    if user_id == 123:
        return {"user_id": "123", "name": "John Doe", "email": "john@123.com"}
    if user_id == 456:
        return {"user_id": "456", "name": "Jane Smith", "email": "jane@123.com"}
    return {"error": "User not found"}

tool1 = StructuredTool.from_function(
    func=user_info,
    name="user_info",
    description="Get user information using user id",
    args_schema=UserQuery
)

In [17]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()

llm = ChatOpenAI(temperature=0)

agent = create_agent(model=llm, tools=[tools[2]])

In [18]:
rahul=agent.invoke({"messages": [("user", "Get info for user 123")]})

In [20]:
rahul

{'messages': [HumanMessage(content='Get info for user 123', additional_kwargs={}, response_metadata={}, id='9c44ffee-31e3-4018-b7aa-2e6839c88d17'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 56, 'total_tokens': 73, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-CwolQYC5vOcvyqsOqpZksfeS2Cmed', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bad02-bd3e-7873-80dd-1d7eaa61734f-0', tool_calls=[{'name': 'get_user_orders', 'args': {'__arg1': '123'}, 'id': 'call_iqXiAOmTSRphKHDcReDIlAIC', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 56, 'output_tokens': 17, 'tota

In [19]:
rahul['messages'][-1].content

'I encountered an error while trying to fetch information for user 123. Please try again later.'

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import create_model
import requests
from typing import Dict, Any, List

def create_custom_api_tool(tool_def: Dict[str, Any]) -> List:
    """
    Convert custom_api tool definition to LangChain tool
    
    Returns:  List containing one LangChain StructuredTool
    """
    name = tool_def["name"]
    description = tool_def["description"]
    api_url = tool_def["api_url"]
    api_request_type = tool_def["api_request_type"]
    custom_message = tool_def.get("custom_message", "")
    input_schema = tool_def["input_schema"]
    
    # Build Pydantic model from input_schema
    fields = {}
    properties = input_schema.get("properties", {})
    required = input_schema.get("required", [])
    
    for field_name, field_spec in properties.items():
        field_type_map = {
            "string": str,
            "number": float,
            "integer":  int,
            "boolean": bool
        }
        field_type = field_type_map.get(field_spec.get("type"), str)
        
        if field_name in required:
            fields[field_name] = (field_type, ...)
        else:
            fields[field_name] = (field_type, None)
    
    # Fallback if no properties
    if not fields: 
        fields = {"_placeholder": (str, None)}
    
    InputModel = create_model(f"{name}_Input", **fields)
    
    # API call function
    def execute_api_call(**kwargs) -> Dict[str, Any]:
        try:
            # Remove placeholder if exists
            kwargs.pop("_placeholder", None)
            
            if api_request_type.upper() == "GET":
                resp = requests.get(api_url, params=kwargs, timeout=10)
            else:  # POST
                resp = requests.post(api_url, json=kwargs, timeout=10)
            
            return {
                "status_code": resp.status_code,
                "data": resp.json() if resp.content else {},
                "custom_message": custom_message
            }
        except Exception as e:
            return {
                "status_code": 500,
                "data": {"error":  str(e)},
                "custom_message": custom_message
            }
    
    # Create LangChain tool
    tool = StructuredTool.from_function(
        func=execute_api_call,
        name=name,
        description=description,
        args_schema=InputModel
    )
    
    return [tool]

In [44]:
tool_def = {
    "id": "agent_f35d529c-5c1a-4bf5-9959-f501cbbdd9b3",
    "type": "custom_api",
    "name": "get_user_orders",
    "description":  "Fetch orders",
    "input_schema": {},
    "custom_message": "take only the order id and provide it to the user",
    "api_url": "http://127.0.0.1:8000/v1/custom-api",
    "api_request_type": "GET"
}

tool_list = create_custom_api_tool(tool_def)
langchain_tool = tool_list[0]
result = langchain_tool.invoke({"user_id": "123"}) 

In [45]:
result

{'status_code': 200,
 'data': [{'id': 'agent_f35d529c-5c1a-4bf5-9959-f501cbbdd9b3',
   'type': 'custom_api',
   'name': 'get_user_orders',
   'description': 'Fetch orders',
   'input_schema': {'type': 'object'},
   'output_schema': {'type': 'object'},
   'custom_message': 'take only the order id and provide it to the user',
   'api_url': 'http://127.0.0.1:8000/v1/custom-api',
   'api_request_type': 'GET',
   'metadata': {'created_at': '2026-01-11T12:44:50.511413+00:00'}}],
 'custom_message': 'take only the order id and provide it to the user'}